# Tool Simulation for Multi-Turn Chatbot Testing

## Overview

In notebooks 01-02, the travel booking agent used hardcoded mock tools — deterministic functions that always return the same data. This works for testing specific scenarios, but mock responses don't adapt to conversation context, and cross-tool consistency is your responsibility.

Strands Evals [`ToolSimulator`](https://github.com/strands-agents/evals/blob/main/src/strands_evals/simulation/tool_simulator.py) solves this with LLM-powered tool responses that are schema-compliant, stateful, and consistent across tools.

> **For a comprehensive deep-dive on ToolSimulator** — including Pydantic output schemas, `share_state_id`, `StateRegistry`, and the full comparison with hardcoded mocks — see **module 04-08 (Tool-Calling Evaluation), Approach 6**. This notebook focuses specifically on combining ToolSimulator with ActorSimulator for **fully simulated multi-turn chatbot conversations**.

## What You'll Learn

1. How to set up ToolSimulator with travel booking tools
2. How to combine `ActorSimulator` + `ToolSimulator` for fully simulated end-to-end multi-turn conversations
3. How stateful tool responses maintain consistency across a multi-turn session

## Architecture

```
┌─────────────────────────────────────────────────────────┐
│                  No Real Anything                        │
│                                                         │
│   ActorSimulator ──> Agent ──> ToolSimulator            │
│   (fake user)        (real)    (fake tools, LLM-powered)│
│                                                         │
│   User messages:     Decisions:    Tool responses:      │
│   LLM-generated      Real model    LLM-generated,       │
│   goal-driven        under test    schema-compliant,     │
│                                    stateful              │
└─────────────────────────────────────────────────────────┘
```

## 1. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from strands import Agent
from strands_evals.simulation.tool_simulator import ToolSimulator
from strands_evals import Case, ActorSimulator
from pydantic import BaseModel, Field
from typing import Optional
import json

print('ToolSimulator imported.')

## 2. Register Travel Booking Tools with ToolSimulator

We register the same travel tools from notebook 01, but this time using `@simulator.tool()`. Each tool gets a Pydantic `output_schema` — the function body is `pass` because the LLM generates responses.

All tools share a single `TRAVEL_STATE` so the simulator maintains consistency: if `search_flights` returns AA101 at $450, `book_flight` will reference that same flight and price.

> **See module 04-08, Approach 6** for detailed explanation of `share_state_id`, `initial_state_description`, and `StateRegistry` mechanics.

In [ ]:
simulator = ToolSimulator()

# --- Output Schemas ---
class FlightSearchResponse(BaseModel):
    flights: list[dict] = Field(description='List of available flights with airline, flight_number, departure, arrival, price, class')
    origin: str = Field(description='Departure city')
    destination: str = Field(description='Arrival city')

class HotelSearchResponse(BaseModel):
    hotels: list[dict] = Field(description='List of available hotels with name, stars, price_per_night, amenities')
    city: str = Field(description='City searched')

class BookingConfirmation(BaseModel):
    booking_ref: str = Field(description='Booking reference number')
    status: str = Field(default='Confirmed', description='Booking status')
    details: str = Field(description='Human-readable booking summary')

class BookingListResponse(BaseModel):
    bookings: list[dict] = Field(description='List of all current bookings')
    total: int = Field(description='Total number of bookings')

class CancellationResponse(BaseModel):
    booking_ref: str = Field(description='Reference of cancelled booking')
    status: str = Field(default='Cancelled', description='New status')
    message: str = Field(description='Cancellation confirmation message')

# --- Register all tools under shared state ---
TRAVEL_STATE = 'travel_bookings'

@simulator.tool(output_schema=FlightSearchResponse, share_state_id=TRAVEL_STATE,
    initial_state_description='Airline booking system with flights to major cities. Prices $300-$1200.')
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> dict:
    """Search for available flights between two cities.
    Args:
        origin: Departure city
        destination: Arrival city
        departure_date: Date in YYYY-MM-DD format
        return_date: Optional return date
    """
    pass

@simulator.tool(output_schema=HotelSearchResponse, share_state_id=TRAVEL_STATE,
    initial_state_description='Hotel system with properties from budget to luxury.')
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> dict:
    """Search for available hotels in a city.
    Args:
        city: Destination city
        check_in: Check-in date
        check_out: Check-out date
        guests: Number of guests
    """
    pass

@simulator.tool(output_schema=BookingConfirmation, share_state_id=TRAVEL_STATE)
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> dict:
    """Book a flight for a passenger.
    Args:
        flight_number: Flight number to book
        passenger_name: Passenger name
        origin: Departure city
        destination: Arrival city
        travel_date: Travel date
    """
    pass

@simulator.tool(output_schema=BookingConfirmation, share_state_id=TRAVEL_STATE)
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> dict:
    """Book a hotel room.
    Args:
        hotel_name: Hotel name
        guest_name: Guest name
        city: City
        check_in: Check-in date
        check_out: Check-out date
    """
    pass

@simulator.tool(output_schema=BookingListResponse, share_state_id=TRAVEL_STATE)
def get_bookings() -> dict:
    """Retrieve all current bookings."""
    pass

@simulator.tool(output_schema=CancellationResponse, share_state_id=TRAVEL_STATE)
def cancel_booking(booking_ref: str) -> dict:
    """Cancel an existing booking.
    Args:
        booking_ref: Booking reference number
    """
    pass

print(f'Registered {len(simulator.list_tools())} simulated tools: {simulator.list_tools()}')

## 3. Fully Simulated Multi-Turn Conversation

This is the key value of combining both simulators for chatbot evaluation: **no real tools, no real users, no scripted scenarios** — yet it tests realistic multi-turn interactions with stateful tool responses.

The `ActorSimulator` generates goal-driven user messages, the agent makes real tool-calling decisions, and the `ToolSimulator` generates consistent, schema-compliant responses. The conversation unfolds naturally.

In [ ]:
SYSTEM_PROMPT = (
    'You are a helpful travel booking assistant. You help users search for flights and hotels, '
    'make bookings, view existing reservations, and cancel bookings. '
    'Always confirm details with the user before completing a booking.'
)

case = Case(
    name='full-trip-planning',
    input='I need to plan a week-long vacation to Paris. Can you help me find flights from New York and a nice hotel?',
    metadata={
        'task_description': (
            'Help the user plan a complete Paris vacation: search for flights from NYC, '
            'compare options, book a flight, search for hotels in Paris for 7 nights, '
            'and book a hotel. The user should end with confirmed bookings for both.'
        )
    }
)

# Clear state for fresh scenario
simulator.state_registry.clear_state(TRAVEL_STATE)

# Create ActorSimulator for user simulation
user_sim = ActorSimulator.from_case_for_user_simulator(case=case, max_turns=6)

# Create agent with simulated tools
agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=[simulator.get_tool(name) for name in simulator.list_tools()],
    callback_handler=None,
)

# Run the fully simulated conversation
user_message = case.input
turn = 0

print(f'Goal: {case.metadata["task_description"]}')
print()

while user_sim.has_next():
    turn += 1
    agent_response = agent(user_message)
    agent_text = str(agent_response)
    
    print(f'--- Turn {turn} ---')
    print(f'  User:  {user_message[:100]}...' if len(user_message) > 100 else f'  User:  {user_message}')
    print(f'  Agent: {agent_text[:150]}...' if len(agent_text) > 150 else f'  Agent: {agent_text}')
    print()
    
    user_result = user_sim.act(agent_text)
    user_message = str(user_result.structured_output.message)

print(f'Conversation completed in {turn} turns.')
print(f'ToolSimulator state has {len(simulator.get_state(TRAVEL_STATE).get("previous_calls", []))} cached tool calls.')

### Inspecting Stateful Consistency

The `StateRegistry` tracks all tool calls and their responses. This is what makes ToolSimulator valuable for multi-turn testing — the LLM generating tool responses has full context of what happened in previous calls, ensuring consistency across the conversation.

In [ ]:
# Inspect what the ToolSimulator generated across the conversation
state = simulator.get_state(TRAVEL_STATE)
print(f'Total tool calls in session: {len(state.get("previous_calls", []))}')
print()
for call in state.get('previous_calls', []):
    print(f'  Tool: {call["tool_name"]}')
    print(f'  Params: {json.dumps(call["parameters"], indent=4)[:150]}...')
    print(f'  Response: {json.dumps(call["response"], indent=4)[:150]}...')
    print()

## Next Steps

- For a comprehensive deep-dive on ToolSimulator patterns (Pydantic schemas, StateRegistry, hardcoded mocks vs. ToolSimulator comparison), see **module 04-08 (Tool-Calling Evaluation), Approach 6**
- Continue to **`04-11-08-e2e-pipeline.ipynb`** to combine all techniques into a production evaluation pipeline